# 🚌 Engine Sound Extractor for OMSI

Extrahuje motorové zvuky z videa pro použití v OMSI Bus Simulator.

**Features:**
- 🎵 AI separace pomocí Demucs (odstranění hlasu/hudby)
- 🎯 AI klasifikace pomocí YAMNet (detekce engine sounds)
- 📊 Frekvenční filtrování
- 🔢 Automatická segmentace podle RPM
- ⚡ GPU akcelerace (10x rychlejší než na CPU!)

---

## 📋 Návod:

1. Klikni **Runtime → Run all** (nebo Ctrl+F9)
2. Nahraj video (nebo použij Google Drive)
3. Počkej na zpracování
4. Stáhni výsledné WAV soubory

## 1️⃣ Instalace závislostí

In [ ]:
%%capture
!pip install librosa soundfile noisereduce scipy moviepy tqdm demucs tensorflow tensorflow-hub

## 2️⃣ Import knihoven

In [ ]:
import os
import sys
from pathlib import Path
from google.colab import files
import zipfile
import shutil

# Download TUI script
!wget -q https://raw.githubusercontent.com/YOUR_REPO/engine_extractor_tui.py -O engine_extractor_tui.py

# Or use uploaded version
print("✓ Knihovny načteny!")

## 3️⃣ Nahraj video

Vyber jednu z možností:
- **A) Upload z počítače** (pomalejší)
- **B) Z Google Drive** (rychlejší pro velké soubory)

In [ ]:
# Možnost A: Upload z počítače
print("📤 Nahraj video soubor...")
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(f"✓ Video nahráno: {video_path}")

In [ ]:
# Možnost B: Z Google Drive (zakomentuj A a odkomentuj B)
# from google.colab import drive
# drive.mount('/content/drive')
# video_path = '/content/drive/MyDrive/video.mp4'  # Změň cestu!
# print(f"✓ Video načteno z Drive: {video_path}")

## 4️⃣ Konfigurace

Nastav parametry zpracování:

In [ ]:
# === KONFIGURACE ===

# Metoda zpracování
METHOD = 'ai_demucs'  # 'classic', 'ai_demucs', 'ai_yamnet'

# AI model (pouze pro Demucs)
AI_MODEL = 'htdemucs'  # 'htdemucs', 'htdemucs_ft', 'mdx_extra'

# Noise reduction úroveň (pouze pro classic)
NOISE_LEVEL = 50  # 0-100

# Frekvenční filtr
FREQ_LOW = 80   # Hz
FREQ_HIGH = 500 # Hz

# Typ motoru
ENGINE_TYPE = 'iveco_cursor8'  # 'fpt_nef6_184', 'iveco_cursor8', 'generic_6cyl'

# Export nastavení
EXPORT_FULL = True      # Exportovat kompletní zvuk?
EXPORT_SEGMENTS = True  # Exportovat segmenty podle RPM?
NUM_SEGMENTS = 5        # Počet segmentů (2-20)

# Výstupní složka
OUTPUT_DIR = '/content/engine_sounds'

print("✓ Konfigurace nastavena!")
print(f"  Metoda: {METHOD}")
print(f"  Motor: {ENGINE_TYPE}")
print(f"  Segmenty: {NUM_SEGMENTS}")

## 5️⃣ Zpracování

**⚠️ DŮLEŽITÉ:** S GPU to bude trvat ~10-20 minut místo 1-2 hodin!

Zkontroluj GPU: **Runtime → Change runtime type → GPU**

In [ ]:
# Check GPU
import torch
if torch.cuda.is_available():
    print(f"✓ GPU dostupné: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ GPU není dostupné! Bude to pomalejší.")
    print("   Runtime → Change runtime type → GPU")

In [ ]:
# Spusť zpracování
import subprocess

cmd = [
    'python', 'engine_extractor_tui.py',
    video_path,
    '-o', OUTPUT_DIR,
    '-m', METHOD,
    '--model', AI_MODEL,
    '-n', str(NOISE_LEVEL),
    '--freq-low', str(FREQ_LOW),
    '--freq-high', str(FREQ_HIGH),
    '-e', ENGINE_TYPE,
    '-s', str(NUM_SEGMENTS)
]

if not EXPORT_FULL:
    cmd.append('--no-full')
if not EXPORT_SEGMENTS:
    cmd.append('--no-segments')

print("🚀 Spouštím zpracování...\n")
subprocess.run(cmd)

## 6️⃣ Stažení výsledků

In [ ]:
# Zabal do ZIP
zip_path = '/content/engine_sounds.zip'
shutil.make_archive('/content/engine_sounds', 'zip', OUTPUT_DIR)

print(f"✓ Výsledky zabaleny do ZIP")
print(f"📦 Velikost: {os.path.getsize(zip_path) / 1024**2:.1f} MB")
print("\n📥 Stahování...")

files.download(zip_path)

In [ ]:
# Nebo zobraz soubory
print("📁 Vygenerované soubory:\n")
for file in sorted(os.listdir(OUTPUT_DIR)):
    if file.endswith('.wav'):
        size = os.path.getsize(os.path.join(OUTPUT_DIR, file)) / 1024**2
        print(f"  {file} ({size:.1f} MB)")

## 7️⃣ Náhled zvuku (volitelné)

Poslechni si výsledky přímo v Colabu:

In [ ]:
import IPython.display as ipd

# Kompletní zvuk
if os.path.exists(os.path.join(OUTPUT_DIR, 'engine_sound_full.wav')):
    print("🔊 Kompletní zvuk:")
    ipd.display(ipd.Audio(os.path.join(OUTPUT_DIR, 'engine_sound_full.wav')))

# První segment
segments = [f for f in os.listdir(OUTPUT_DIR) if f.startswith('engine_') and f.endswith('rpm.wav')]
if segments:
    print(f"\n🔊 První segment ({segments[0]}):")
    ipd.display(ipd.Audio(os.path.join(OUTPUT_DIR, segments[0])))

---

## ✅ Hotovo!

**Co dál:**
1. Stáhni ZIP soubor
2. Rozbal ho
3. Zkopíruj WAV soubory do `Vehicles\[tvůj_bus]\sound\`
4. Nastav v `sound.cfg`:

```ini
[engine]
idle=engine_600rpm.wav
low=engine_1200rpm.wav
medium=engine_1800rpm.wav
high=engine_2400rpm.wav
```

---

**Užij si realističtější zvuky v OMSI! 🚌🎵**